## **4)— Visualising Distributions and Testing for Statistical Significance**

### Introduction

At this stage, we move beyond simple averages and strengthen our argument using:

* Distribution visualisation (box plots, histograms, KDE)
* Statistical hypothesis testing (t-test)

The goal is to determine whether the reduction in death rates after handwashing is:

* Just a random fluctuation
* Or statistically significant

---

### Challenge 1 — Difference in Average Monthly Death Rate

We compare the **mean monthly death rate** before and after handwashing.

#### Calculate the Means

```python
avg_prob_before = before_washing.pct_deaths.mean() * 100
print(f'Chance of death during childbirth BEFORE handwashing: {avg_prob_before:.3f}%')

avg_prob_after = after_washing.pct_deaths.mean() * 100
print(f'Chance of death during childbirth AFTER handwashing: {avg_prob_after:.3f}%')
```

#### Compute the Difference

```python
mean_diff = avg_prob_before - avg_prob_after
print(f'Handwashing reduced the monthly proportion of deaths by {mean_diff:.3f}%')

times = avg_prob_before / avg_prob_after
print(f'This is a {times:.2f}x improvement!')
```

#### Interpretation

* Before handwashing: ~10.5%
* After handwashing: ~2.1%
* Reduction: ~8.4 percentage points
* Roughly **5x improvement**

This is not a small effect — it is dramatic.

---

### Challenge 2 — Box Plots to Compare Distributions

Averages are useful, but distributions tell a deeper story.

#### Step 1: Create a Category Column Using `np.where()`

```python
df_monthly['washing_hands'] = np.where(
    df_monthly.date < handwashing_start,
    'No',
    'Yes'
)
```

This creates a categorical column indicating whether handwashing was mandatory.

---

#### Step 2: Create a Box Plot with Plotly

```python
box = px.box(
    df_monthly,
    x='washing_hands',
    y='pct_deaths',
    color='washing_hands',
    title='How Have the Stats Changed with Handwashing?'
)

box.update_layout(
    xaxis_title='Washing Hands?',
    yaxis_title='Percentage of Monthly Deaths'
)

box.show()
```

#### What the Box Plot Shows

* Lower median after handwashing
* Lower maximum
* Lower upper quartile (Q3)
* Reduced overall spread

The reduction is visible not just in the mean, but across the entire distribution.

---

### Challenge 3 — Histograms for Monthly Distribution

To compare frequency patterns, we build overlapping histograms.

#### Key Considerations

* Use `color='washing_hands'`
* Use `histnorm='percent'` (important because periods are different lengths)
* Use `opacity` for overlap visibility
* Optionally add `marginal='box'`

```python
hist = px.histogram(
    df_monthly,
    x='pct_deaths',
    color='washing_hands',
    nbins=30,
    opacity=0.6,
    barmode='overlay',
    histnorm='percent',
    marginal='box'
)

hist.update_layout(
    xaxis_title='Proportion of Monthly Deaths',
    yaxis_title='Percent'
)

hist.show()
```

#### Interpretation

* Before handwashing: wide, heavy right tail (high death rates)
* After handwashing: tight cluster at low percentages
* Far fewer extreme months

The distributions barely overlap in the dangerous range.

---

### Challenge 4 — Kernel Density Estimate (KDE)

Histograms are jagged due to small sample size (~98 months).

KDE provides a smooth estimate of the underlying distribution.

#### Initial KDE

```python
plt.figure(dpi=200)

sns.kdeplot(before_washing.pct_deaths, fill=True)
sns.kdeplot(after_washing.pct_deaths, fill=True)

plt.title('Est. Distribution of Monthly Death Rate Before and After Handwashing')
plt.show()
```

Problem:
The curve extends into negative death rates (which is impossible).

---

#### Fix Using `clip`

```python
plt.figure(dpi=200)

sns.kdeplot(before_washing.pct_deaths, fill=True, clip=(0,1))
sns.kdeplot(after_washing.pct_deaths, fill=True, clip=(0,1))

plt.title('Est. Distribution of Monthly Death Rate Before and After Handwashing')
plt.xlim(0, 0.40)

plt.show()
```

#### Interpretation

* Before handwashing: distribution centered high, wide spread
* After handwashing: narrow, tightly concentrated near zero
* Clear separation between curves

---

### Challenge 5 — T-Test for Statistical Significance

Now we formally test whether the difference in means is statistically significant.

We use an **independent samples t-test**.

#### Step 1: Import Stats

```python
import scipy.stats as stats
```

#### Step 2: Run the Test

```python
t_stat, p_value = stats.ttest_ind(
    a=before_washing.pct_deaths,
    b=after_washing.pct_deaths
)

print(f'p-value is {p_value:.10f}')
print(f't-statistic is {t_stat:.4f}')
```

#### Interpretation

* p-value ≈ 0.0000002985
* Far below 0.01 (1%)

This means we can be more than **99% confident** that the reduction in death rates is not due to chance.

The difference is statistically significant.

---

### Final Takeaways

* The average death rate dropped dramatically.
* The entire distribution shifted downward.
* Extreme months became rare.
* KDE shows strong separation between periods.
* The t-test confirms statistical significance.

This is no longer just visual evidence — it is statistically validated evidence that handwashing saved lives.
